# Chapter 31: Camera Models

<a href="../lite/lab/index.html?path=ch31_camera_models.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Hold your phone camera up. Every pixel on the screen is a ray shooting out from the lens
into the 3D world. The **camera model** is the mathematical description of which 3D points
land on which pixels. Get it right, and you can measure real world distances from photos.
Get it wrong, and your visual SLAM will fail silently.

## 31.1 Projection

The **pinhole camera model** projects a 3D point $\mathbf{P} = [X, Y, Z]^T$ to a 2D pixel:

$$\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = \frac{1}{Z} K \begin{bmatrix} X \\ Y \\ Z \end{bmatrix}$$

where $K$ is the **intrinsic matrix**:

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
fx, fy = 500, 500         # focal length (pixels)
cx, cy = 320, 240         # principal point (image center)
img_w, img_h = 640, 480   # image size

# 3D cube corners
cube_size = 1.0
cube_center = np.array([0, 0, 5])   # cube is 5 meters in front of camera
# ──────────────────────────────────────────────────────────────────────────────

K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

# Generate cube vertices
offsets = np.array([[dx, dy, dz] for dx in [-1,1] for dy in [-1,1] for dz in [-1,1]]) * cube_size/2
points_3d = cube_center + offsets

# Project to 2D
projected = (K @ points_3d.T).T
pixels = projected[:, :2] / projected[:, 2:3]

# Cube edges
edges = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),(4,5),(4,6),(5,7),(6,7)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.set_title("3D Scene (top view)", fontsize=13)
ax.scatter(points_3d[:, 0], points_3d[:, 2], c='steelblue', s=60, zorder=5)
for i, j in edges:
    ax.plot([points_3d[i,0], points_3d[j,0]], [points_3d[i,2], points_3d[j,2]], 'steelblue', lw=1)
ax.plot(0, 0, 'ko', ms=10, label='camera')
ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)"); ax.legend()

ax = axes[1]
ax.set_title("Projected image (pinhole)", fontsize=13)
ax.scatter(pixels[:, 0], pixels[:, 1], c='tomato', s=60, zorder=5)
for i, j in edges:
    ax.plot([pixels[i,0], pixels[j,0]], [pixels[i,1], pixels[j,1]], 'tomato', lw=1.5)
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)  # image coordinates: y flipped
ax.set_xlabel("u (pixels)"); ax.set_ylabel("v (pixels)")
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print(f"Intrinsic matrix K:\n{K}")

## 31.2 Intrinsics

The intrinsic matrix $K$ encodes the camera's internal properties:
- **Focal length** ($f_x, f_y$): how much the lens magnifies the scene
- **Principal point** ($c_x, c_y$): where the optical axis hits the image sensor

## 31.3 Extrinsics

The **extrinsic matrix** $[R | t]$ describes the camera's pose (position and orientation)
in the world frame. The full projection is:

$$\mathbf{p} = K [R | t] \mathbf{P}_{world}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
cam_yaw_deg = 15.0      # camera rotation about Y axis (try 0, 15, 45)
cam_x = 1.0             # camera X offset (try 0, 1, 3)
# ──────────────────────────────────────────────────────────────────────────────

yaw = np.radians(cam_yaw_deg)
R_cam = np.array([[np.cos(yaw), 0, np.sin(yaw)],
                   [0, 1, 0],
                   [-np.sin(yaw), 0, np.cos(yaw)]])
t_cam = np.array([cam_x, 0, 0])

# Transform world points to camera frame
points_cam = (R_cam @ (points_3d - t_cam).T).T
visible = points_cam[:, 2] > 0.1
projected2 = (K @ points_cam[visible].T).T
pixels2 = projected2[:, :2] / projected2[:, 2:3]

fig, ax = plt.subplots(figsize=(8, 6))
valid = (pixels2[:, 0] >= 0) & (pixels2[:, 0] < img_w) & (pixels2[:, 1] >= 0) & (pixels2[:, 1] < img_h)
ax.scatter(pixels2[valid, 0], pixels2[valid, 1], c='steelblue', s=60, zorder=5)
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
ax.set_xlabel("u (pixels)"); ax.set_ylabel("v (pixels)")
ax.set_title(f"View from rotated camera (yaw={cam_yaw_deg}°, x={cam_x}m)", fontsize=13)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 31.4 Calibration

Camera calibration estimates $K$ from known 3D to 2D correspondences.
In practice, a checkerboard pattern provides these correspondences.

**Key observations:**
- The pinhole model is a good approximation for most cameras.
- Real lenses add **distortion** (barrel, pincushion) that must be corrected.
- Calibration is a one time step but is critical for all downstream visual SLAM.

---

## Exercises

### Exercise 31.1
Project a 3D triangle (vertices at [0,0,3], [1,0,3], [0.5,1,3]) using a pinhole camera
with $f = 400$, $c = (320, 240)$. Plot the projected triangle in the image.

### Exercise 31.2 (challenge)
Given 6 known 3D to 2D correspondences, estimate the camera intrinsic matrix K using
least squares. Compare with the true K.

In [ ]:
# Your code here